In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_log_error
import matplotlib.pyplot as plt
import numpy as np
import shap
import optuna
import os

os.makedirs("images", exist_ok=True)

## Data Loading

Load Kaggle House Price

In [ ]:
df = pd.read_csv(
    'data/train.csv',
    encoding='latin1'
)

print(df.shape)
df.head()

## Missing Value Handling

Missing values were handled based on the meaning of each feature.

- LotFrontage was imputed using the median value.
- Garage-related features were filled with "None" when the house had no garage.
- Basement-related features were handled similarly.
- Electrical was imputed using the mode.

In [ ]:
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
df['GarageType'] = df['GarageType'].fillna('None')
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)
df['GarageFinish'] = df['GarageFinish'].fillna('None')
df['GarageQual'] = df['GarageQual'].fillna('None')
df['GarageCond'] = df['GarageCond'].fillna('None')
df['PoolQC'] = df['PoolQC'].fillna('None')
df['Fence'] = df['Fence'].fillna('None')
df['MiscFeature'] = df['MiscFeature'].fillna('None')
df['FireplaceQu'] = df['FireplaceQu'].fillna('None')
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
df['BsmtQual'] = df['BsmtQual'].fillna('None')
df['BsmtCond'] = df['BsmtCond'].fillna('None')
df['BsmtFinType1'] = df['BsmtFinType1'].fillna('None')
df['Alley'] = df['Alley'].fillna('None')
mas_mode = df["MasVnrType"].mode()[0]

df.loc[(df["MasVnrType"].isnull()) & (df["MasVnrArea"] > 0), "MasVnrType"] = mas_mode

df["MasVnrType"] = df["MasVnrType"].fillna("None")

bsmt_mode = df["BsmtFinType2"].mode()[0]

df.loc[(df["BsmtFinType2"].isnull()) & (df["BsmtFinSF2"] > 0), "BsmtFinType2"] = bsmt_mode

df["BsmtFinType2"] = df["BsmtFinType2"].fillna("None")

df.loc[(df['BsmtExposure'].isnull()) & (df['TotalBsmtSF'] > 0), 'BsmtExposure'] = 'No'
df['BsmtExposure'] = df['BsmtExposure'].fillna('None')

## Feature Selection

Based on the EDA findings, an initial feature set consisting of 10 numerical and 10 categorical features was constructed.

In [ ]:
result = []

features = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', 
            '1stFlrSF', 'FullBath', 'TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd',
            'ExterQual', 'Neighborhood', 'KitchenQual', 'BsmtQual', 'FireplaceQu', 
            'SaleCondition', 'BsmtExposure', 'GarageType', 'BsmtCond', 'MSZoning']

## Baseline Model: Linear Regression

Linear Regression was used as the baseline model. The target variable was transformed using log1p, and predictions were converted back to the original scale using expm1 for evaluation.

In [ ]:
X = df[features]
y = np.log1p(df['SalePrice'])

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression(fit_intercept=True))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)

pred_price = np.expm1(pred)
y_test_price = np.expm1(y_test)
pred_price = np.maximum(pred_price, 0)

result.append({
    'model': 'LinearRegression',
    'MAE': mean_absolute_error(y_test_price, pred_price),
    'RMSE': root_mean_squared_error(y_test_price, pred_price),
    'RMSLE': root_mean_squared_log_error(y_test_price, pred_price),
    'R2_Score': r2_score(y_test_price, pred_price)
})

## LightGBM Model

LightGBM was selected as a tree-based ensemble model and tuned using GridSearchCV.

In [ ]:
X = df[features]
y = np.log1p(df['SalePrice'])

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown = 'ignore'), cat_features),
    ('num', StandardScaler(), num_features)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LGBMRegressor(random_state = 42, verbose = -1))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

params = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__n_estimators": [200, 500],
    "model__num_leaves": [31, 63],
    "model__colsample_bytree": [0.8, 1.0]
}

grid = GridSearchCV(pipe, params, cv=5, scoring='r2')

grid.fit(X_train, y_train)

pred = grid.predict(X_test)

pred_price = np.expm1(pred)
y_test_price = np.expm1(y_test)

result.append({
    'model': 'LGBMRegressor',
    'MAE': mean_absolute_error(y_test_price, pred_price),
    'RMSE': root_mean_squared_error(y_test_price, pred_price),
    'RMSLE': root_mean_squared_log_error(y_test_price, pred_price),
    'R2_Score': r2_score(y_test_price, pred_price)
})

## XGBoost Model

XGBoost was trained and tuned using GridSearchCV. Since XGBoost achieved the best RMSLE among the tested models, it was selected as the final model candidate.

In [ ]:
X = df[features]
y = np.log1p(df['SalePrice'])

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42, eval_metric='rmse'))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

params = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5, 7],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}
    
grid = GridSearchCV(pipe, params, cv=5, scoring='r2', n_jobs=-1)

grid.fit(X_train, y_train)

pred = grid.predict(X_test)

pred_price = np.expm1(pred)
y_test_price = np.expm1(y_test)

pred_price = np.maximum(pred_price, 0)

result.append({
    'model': 'XGBRegressor',
    'MAE': mean_absolute_error(y_test_price, pred_price),
    'RMSE': root_mean_squared_error(y_test_price, pred_price),
    'RMSLE': root_mean_squared_log_error(y_test_price, pred_price),
    'R2_Score': r2_score(y_test_price, pred_price)
})

## Model Comparison

The performance of Linear Regression, LightGBM, and XGBoost was compared using MAE, RMSE, RMSLE, and R².

In [ ]:
result_df = pd.DataFrame(result)
result_df

### Result

XGBoost achieved the lowest RMSLE score among the tested models. Since RMSLE is the official evaluation metric used in the Kaggle House Prices competition, XGBoost was selected as the final model for further analysis.

## Feature Importance Analysis

Feature importance scores from the selected XGBoost model were analyzed to identify the most influential variables.

In [ ]:
best_xgb = grid.best_estimator_

importance = pd.DataFrame({
    'feature': best_xgb.named_steps['preprocessor'].get_feature_names_out(),
    'importance': best_xgb.named_steps['model'].feature_importances_
}).sort_values("importance", ascending=False)

importance.head(20)

### Observation

OverallQual, GrLivArea, GarageCars, and several location-related features appeared among the most important variables. Based on these results, additional feature selection experiments were conducted using the Top10 and Top15 features.

In [ ]:
top20 = importance.head(20)

plt.figure(figsize=(8, 6))
plt.barh(top20["feature"], top20["importance"])
plt.gca().invert_yaxis()
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.savefig(
    "images/feature_importance.png",
    bbox_inches="tight"
)
plt.close()

## Feature Selection Experiment

Based on the XGBoost feature importance scores, two reduced feature sets (Top10 and Top15) were created and compared with the initial feature set.

In [ ]:
feature_sets = {
    'Initial': ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'FullBath', 'TotRmsAbvGrd', 'YearBuilt', 'YearRemodAdd',
                'ExterQual', 'Neighborhood', 'KitchenQual', 'BsmtQual', 'FireplaceQu', 'SaleCondition', 'BsmtExposure', 'GarageType', 'BsmtCond', 'MSZoning'],
    'Top10': ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'FullBath', 'ExterQual', 'Neighborhood', 'KitchenQual', 'FireplaceQu', 'BsmtQual'], 
    'Top15': ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', '1stFlrSF', 'FullBath', 'YearBuilt', 'YearRemodAdd', 'ExterQual', 'Neighborhood', 
                'KitchenQual', 'BsmtQual', 'FireplaceQu', 'BsmtExposure', 'MSZoning']
}

### Experiment Setup

Optuna was used to tune XGBoost hyperparameters for each feature set.

In [ ]:
result = []
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    model = XGBRegressor(
        n_estimators = trial.suggest_int('n_estimators', 100, 500),
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2),
        max_depth = trial.suggest_int('max_depth', 3, 8),
        subsample = trial.suggest_float('subsample', 0.7, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 1.0),
        random_state = 42,
        eval_metric='rmse'
    )

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    score = cross_val_score(
       pipe,
       X_train,
       y_train,
       cv = 5,
       scoring='r2'
    ).mean()

    return score


for feature_name, features in feature_sets.items():
    X = df[features]
    y = np.log1p(df['SalePrice'])

    cat_features = X.select_dtypes(include='object').columns.tolist()
    num_features = X.select_dtypes(exclude='object').columns.tolist()

    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', StandardScaler(), num_features)
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

    sampler=optuna.samplers.TPESampler(seed=42)

    study = optuna.create_study(direction='maximize', sampler = sampler)
    study.optimize(objective, n_trials = 50)

    print(study.best_params)
    print(study.best_value)

    best_params = study.best_params

    best_model = XGBRegressor(
        **best_params,
        random_state = 42,
        eval_metric = 'rmse'
    )

    best_pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', best_model)
    ])

    best_pipe.fit(X_train, y_train)

    pred = best_pipe.predict(X_test)

    pred_price = np.expm1(pred)
    y_test_price = np.expm1(y_test)

    result.append({
        'feature': feature_name,
        'model': 'XGBRegressor',
        'MAE': mean_absolute_error(y_test_price, pred_price),
        'RMSE': root_mean_squared_error(y_test_price, pred_price),
        'RMSLE': root_mean_squared_log_error(y_test_price, pred_price),
        'R2_Score': r2_score(y_test_price, pred_price)
    })
    
    if feature_name == "Top15":
        final_pipe = best_pipe
        final_X_train = X_train

result_df = pd.DataFrame(result)
result_df

### Result

Top10 showed a noticeable performance decrease.

Top15 achieved performance close to the Initial feature set while reducing the number of features.

Therefore, Top15 was selected as the final feature set.

## SHAP Analysis

SHAP values were used to interpret the predictions of the final XGBoost model and identify the most influential features.

In [ ]:
xgb_model = final_pipe.named_steps['model']
preprocessor = final_pipe.named_steps['preprocessor']

X_train_transformed = preprocessor.transform(final_X_train).toarray()
feature_names = preprocessor.get_feature_names_out()

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train_transformed)

X_train_shap = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

shap.summary_plot(
    shap_values,
    X_train_shap,
    plot_type='bar'
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_train_shap,
    show=False
)

plt.savefig(
    "images/shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

### Observation

OverallQual, GrLivArea, GarageCars, and YearBuilt showed the largest impact on model predictions.

The SHAP analysis was largely consistent with the feature importance results and the findings from the EDA process.

Features with higher values generally contributed positively to house prices, while lower values tended to decrease predicted prices.

# Final Conclusion

Three regression models (Linear Regression, LightGBM, and XGBoost) were evaluated for house price prediction.

Among the tested models, XGBoost achieved the lowest RMSLE score, which is the official evaluation metric of the Kaggle House Prices competition. Therefore, XGBoost was selected as the final model.

Feature importance analysis and SHAP interpretation were performed to identify the most influential variables. Based on the feature importance results, Top10 and Top15 feature sets were evaluated.

The Top10 feature set showed a noticeable performance decrease, while the Top15 feature set maintained performance close to the initial feature set with fewer variables. Therefore, the Top15 feature set was selected.

The final model consisted of:

- XGBoost Regressor
- Optuna Hyperparameter Tuning
- log1p Target Transformation
- Top15 Feature Set

The results showed that EDA findings, feature importance scores, and SHAP values were largely consistent, indicating that the selected features captured the major factors affecting house prices.